<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 실습 1. Zero-shot / One-shot / Few-shot 비교

### ① 라이브러리 설치

In [ ]:
# Hugging Face의 transformers 라이브러리를 설치합니다.
# -q 옵션은 설치 메시지를 간단하게 표시합니다.
!pip -q install transformers accelerate sentencepiece

### ② 모델 불러오기
한국어 instruction-following이 가능한 비교적 작은 모델을 사용합니다. Colab GPU 사용을 권장합니다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 사용할 모델 이름
# Colab에서 실행하기 쉽도록 비교적 작은 모델을 사용합니다.
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 문장을 토큰으로 변환하는 tokenizer를 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 사전학습된 언어모델을 불러옵니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("모델 로딩 완료!")

### ③ LLM에게 질문하는 함수

In [ ]:
def ask_llm(prompt):
    """
    prompt를 모델에 전달하고
    생성된 답변만 반환하는 함수
    """

    # 채팅 모델이 이해할 수 있는 message 형식으로 만듭니다.
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 모델의 chat template에 맞는 문자열로 변환합니다.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 문자열을 모델이 처리할 수 있는 token ID로 변환합니다.
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 모델이 답변을 생성합니다.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,

            # 최대 30개의 새 토큰을 생성합니다.
            max_new_tokens=30,

            # 실험 결과를 쉽게 비교하기 위해
            # 무작위 sampling을 사용하지 않습니다.
            do_sample=False
        )

    # 입력 부분을 제외하고 새로 생성된 답변만 가져옵니다.
    generated = outputs[0][inputs["input_ids"].shape[1]:]

    # token ID를 다시 사람이 읽을 수 있는 문자열로 변환합니다.
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer.strip()

### 실험 A. Zero-shot
예제를 하나도 보여주지 않고 바로 문제를 해결하게 합니다.

In [ ]:
review = "배송은 조금 늦었지만 제품의 품질은 정말 만족스럽습니다."

zero_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

리뷰: {review}

감정:
"""

result = ask_llm(zero_shot_prompt)

print("=== Zero-shot ===")
print(result)

### 실험 B. One-shot
이번에는 모델에게 정답 예제 1개를 먼저 보여줍니다.

In [ ]:
one_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시]

리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(one_shot_prompt)

print("=== One-shot ===")
print(result)

### 실험 C. Few-shot
이번에는 서로 다른 유형의 예제 3개를 제공합니다.

In [ ]:
few_shot_prompt = f"""
다음 상품 리뷰의 감정을 분류하세요.

감정은 반드시 다음 중 하나만 선택하세요.
긍정
중립
부정

[예시 1]
리뷰: 가격도 저렴하고 성능도 매우 좋습니다.
감정: 긍정

[예시 2]
리뷰: 특별히 좋지도 나쁘지도 않습니다.
감정: 중립

[예시 3]
리뷰: 절대 다시 사고 싶지 않습니다.
감정: 부정

[새로운 문제]

리뷰: {review}
감정:
"""

result = ask_llm(few_shot_prompt)

print("=== Few-shot ===")
print(result)

## 실습 2. 좋은 프롬프트 구조 만들기

### 실습용 문서 준비

In [ ]:
document = """
대규모 언어 모델(LLM)은 방대한 텍스트 데이터로 학습된
인공지능 모델이다. 사용자의 질문에 답하거나 문서를 요약하고,
글을 작성하거나 프로그램 코드를 생성할 수 있다.

최근에는 LLM이 외부 문서를 검색하는 RAG,
외부 도구를 사용하는 Tool Calling,
여러 작업을 스스로 수행하는 AI Agent와 결합되고 있다.

하지만 LLM은 사실과 다른 내용을 생성하는 환각(Hallucination),
프롬프트 인젝션, 개인정보 유출 등의 문제를 일으킬 수 있다.
따라서 실제 시스템에서는 모델의 출력뿐 아니라
데이터, 권한, 도구 사용에 대한 검증도 중요하다.
"""

### 실험 A. 모호한 프롬프트

In [ ]:
prompt_a = f"""
이 문서를 잘 요약해줘.

문서:
{document}
"""

result_a = ask_llm(prompt_a)

print("=== A. 모호한 프롬프트 ===")
print(result_a)

### 실험 B. Role 추가
이번에는 모델에게 어떤 관점에서 작업할 것인지 알려줍니다.

In [ ]:
prompt_b = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 교수다.

[Task]
다음 문서를 요약하라.

[Document]
{document}
"""

result_b = ask_llm(prompt_b)

print("=== B. Role 추가 ===")
print(result_b)

### 실험 C. Role + Context + Task
이제 누구를 위한 결과인지와 정확히 무엇을 해야 하는지를 추가합니다.

In [ ]:
prompt_c = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 교수다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Document]
{document}
"""

result_c = ask_llm(prompt_c)

print("=== C. Role + Context + Task ===")
print(result_c)

### 실험 D. Constraints 추가
이제 모델이 지켜야 할 조건을 명시합니다.

In [ ]:
prompt_d = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Document]
{document}
"""

result_d = ask_llm(prompt_d)

print("=== D. Constraints 추가 ===")
print(result_d)

### 실험 E. Example 추가
이번에는 원하는 출력의 모습을 예제로 보여줍니다.

In [ ]:
prompt_e = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Example]
개념: 자연어처리(Natural Language Processing)
설명: 컴퓨터가 인간의 언어를 처리하는 AI 분야이다.

[Document]
{document}
"""

result_e = ask_llm(prompt_e)

print("=== E. Example 추가 ===")
print(result_e)

### 실험 F. Output Format 추가
마지막으로 출력 형식을 명확하게 지정합니다.

In [ ]:
prompt_f = f"""
[Role]
너는 컴퓨터공학 전공 학생을 가르치는 조교다.

[Context]
이 요약은 컴퓨터공학과 4학년 학생의
지능시스템 수업 복습 자료로 사용한다.

[Task]
다음 문서에서 핵심 개념을 추출하여 요약하라.

[Constraints]
- 핵심 개념은 3개만 선택한다.
- 각 설명은 2문장 이내로 작성한다.
- 문서에 없는 사실을 추가하지 않는다.
- 중요한 전문용어는 영어를 함께 표시한다.

[Example]
개념: 자연어처리(Natural Language Processing)
설명: 컴퓨터가 인간의 언어를 처리하는 AI 분야이다.

[Output Format]
Markdown 표로 작성한다.

열은 다음 세 개만 사용한다.

| 개념 | 설명 | 중요 이유 |

[Document]
{document}
"""

result_f = ask_llm(prompt_f)

print("=== F. 6요소 완성 ===")
print(result_f)